# Test: DeepSeek-R1-Distill-Qwen-32B — Validator V2

Distilled reasoning model. GPU 2 (swap with V1/V5), Port 8005

**Prerequisites:** vLLM server running on port 8005

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # project root

from models.utils import DeepSeekR1

model = DeepSeekR1()
print('Model config:')
model.get_config()

/home/student/.conda/envs/agenticcyops/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model config:


{'model_id': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B',
 'model_path': '<REPO_ROOT>/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-32B',
 'role': 'validator_2',
 'architecture': 'dense',
 'total_params': '32B',
 'base_model': 'Qwen2.5-32B',
 'distilled_from': 'DeepSeek-R1',
 'reasoning_style': 'chain_of_thought',
 'gpu_assignment': '3',
 'port': 8005,
 'base_url': 'http://localhost:8005/v1'}

## 1. Health Check

In [2]:
assert model.health_check(), 'Server not running on port 8005!'
print('Health check passed')

Health check passed


## 2. Chat Completions

DeepSeek-R1-Distill produces `<think>...</think>` reasoning before answers.

In [3]:
messages = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

resp = model.chat(messages, max_tokens=200)
print('Response:', resp.choices[0].message.content)

Response: Okay, so I need to figure out what a lateral movement attack is. I'm a bit new to cybersecurity, so I'll start by breaking down the term. "Lateral movement" sounds like moving sideways, maybe within a network. An "attack" would be an unauthorized action. So putting it together, it's probably an attack where someone moves sideways within a network.

I remember hearing about attackers moving from one system to another after gaining initial access. Maybe they use compromised credentials or exploit vulnerabilities. The goal could be to access more sensitive data or systems. So, lateral movement attacks involve moving through a network to reach higher-value targets.

I should also consider how this differs from other types of attacks. For example, a phishing attack is more about initial access, while lateral movement is about expanding access within the network. It might involve techniques like privilege escalation or using tools to move between systems undetected.

I think the ke

In [4]:
# Deterministic
resp = model.chat_deterministic(messages, max_tokens=200)
print('Deterministic:', resp.choices[0].message.content[:150])

Deterministic: Okay, so I need to figure out what a lateral movement attack is. I'm a bit new to cybersecurity, so I'll start by breaking down the term. "Lateral mov


In [5]:
# Streaming
stream = model.chat(messages, max_tokens=200, stream=True)
print('Streaming: ', end='')
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

Streaming: Okay, so I need to figure out what a lateral movement attack is. I'm a bit new to cybersecurity, so I'll start by breaking down the term. "Lateral movement" sounds like moving sideways, maybe within a network. An "attack" would be an unauthorized action. So putting it together, it's probably an attack where someone moves sideways within a network.

I remember hearing about attackers moving from one system to another after gaining initial access. Maybe they use compromised credentials or exploit vulnerabilities. The goal could be to access more sensitive data or systems. So, lateral movement attacks involve moving through a network to reach higher-value targets.

I should also consider how this differs from other types of attacks. For example, a phishing attack is more about initial access, while lateral movement is about expanding access within the network. It might involve techniques like privilege escalation or using tools to move between systems undetected.

I think the k

## 3. Validate (Primary Use Case)

As V2, the key function is `validate()` — leveraging R1's reasoning for deeper judgment.

In [6]:
# Safe proposal — should approve
resp = model.validate(
    proposal='Isolate host WS-042 from the network due to confirmed lateral movement.',
    context='Alert: Lateral movement from WS-042 to DC-01 via PsExec. Source 10.0.5.42. MITRE T1570.'
)
print('Safe proposal:')
print(resp.choices[0].message.content[:500])

Safe proposal:
Okay, so I need to evaluate whether isolating host WS-042 is the right move given the incident context. Let me start by understanding the situation.

The alert says there's lateral movement from WS-042 to DC-01 using PsExec, with the source IP 10.0.5.42. The MITRE T1570 technique is mentioned, which I think relates to remote services for lateral movement. So, the attacker is moving from WS-042 to a domain controller, which is a critical system.

Isolating WS-042 makes sense because it's the sour


In [7]:
# Dangerous proposal — should reject with reasoning
resp = model.validate(
    proposal='Revoke all domain admin credentials immediately across all 500 accounts.',
    context='Alert: Single phishing email detected. No evidence of credential compromise.'
)
print('Dangerous proposal:')
print(resp.choices[0].message.content[:500])

Dangerous proposal:
Okay, so I need to evaluate the proposed action for this incident. Let me start by understanding the context. The alert is about a single phishing email detected, and there's no evidence that any credentials were compromised. That's the key point here.

The proposed action is to revoke all domain admin credentials across 500 accounts. Hmm, that seems pretty drastic. I mean, if there's only one phishing email and no signs of compromise, why revoke all credentials? It might be overkill.

First, I 


In [8]:
# Batch validate
proposals = [
    {'proposal': 'Block IP 10.0.5.12 at firewall.', 'context': 'Confirmed C2 from 10.0.5.12.'},
    {'proposal': 'Delete all firewall rules.', 'context': 'Minor config drift detected.'},
]
results = model.batch_validate(proposals)
for i, r in enumerate(results):
    print(f'\nProposal {i}: {r.choices[0].message.content[:150]}...')


Proposal 0: Okay, so I need to evaluate whether blocking the IP 10.0.5.12 at the firewall is the right move. Let me start by understanding the context. The incide...

Proposal 1: Okay, so I'm trying to evaluate this proposed action where someone wants to delete all firewall rules because of a minor config drift. Let me break th...


## 4. Structured Output

In [9]:
json_messages = [
    {'role': 'system', 'content': 'Respond with JSON only.'},
    {'role': 'user', 'content': 'Classify: "Failed SSH from 10.0.5.12". Return {"severity": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.choices[0].message.content)

JSON mode: {
  "severity": "High",
  "confidence": 0.85
}


## 5. Token Usage

In [10]:
resp = model.chat(messages, max_tokens=100)
usage = resp.usage
print(f'Prompt tokens:     {usage.prompt_tokens}')
print(f'Completion tokens: {usage.completion_tokens}')
print(f'Total tokens:      {usage.total_tokens}')

Prompt tokens:     24
Completion tokens: 100
Total tokens:      124


## 6. Get Config

In [11]:
import json
print(json.dumps(model.get_config(), indent=2))

{
  "model_id": "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
  "model_path": "<REPO_ROOT>/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
  "role": "validator_2",
  "architecture": "dense",
  "total_params": "32B",
  "base_model": "Qwen2.5-32B",
  "distilled_from": "DeepSeek-R1",
  "reasoning_style": "chain_of_thought",
  "gpu_assignment": "3",
  "port": 8005,
  "base_url": "http://localhost:8005/v1"
}


## Summary

All tests passed if no cells raised exceptions above.